# MULTIVERSES (Will take a very long time to run locally)

In [1]:
# Import libraries
import os
import pandas as pd
import numpy as np
import time
import statsmodels.api as sm
from scipy.stats import norm
import gc
from tools import *

## Change input paths here

In [ ]:
# Data paths
EMA_path = "Z:\Projects\EMA_Project\Data\EMA_UCLA_Data"
EMA_data_file = "ERT_EMA_DailySurvey.csv"
Data_path = f"{EMA_path}/{EMA_data_file}"

Intake_data_file = "ERT_EMA_Intake.csv"
Data_path2 = f"{EMA_path}/{Intake_data_file}"

## DIMENSION LABELS

In [ ]:
definition_transition_matrix = ["Transition", "Proportion"]
distance_measure = ["Pearson", "Spearman", "Euclid"]
comparison_method = ["Pairwise", "Leave-One-Subject-Out"]
control_for_entropy = ["Entropy", "No_Entropy"]
filtering = ["No_Filtering", "More_Than_10_Responses", "More_Than_20_Seconds", "MT10R_AND_MT20S"]
temporal_overlap = ["Start_Date", "Response_Window", "Date_and_Time", "No_Time_Control"]
control_for_demo = ["Demographics", "No_Demographics"]
baseline_ERS_use = ["Baseline_ER", "No_Baseline_ER"]

## RESULTS SETUP AND DATA IMPORT

In [ ]:
# Empty lists for results and analytic dimensions
PA_results_list = []
NA_results_list = []
ST_results_list = []
Analytic_dimensions = []

# Import of data
EMA_data_new = pd.read_csv(Data_path)
EMA_data_new.rename(columns={'Duration (in seconds)': 'duration'}, inplace=True)
EMA_data_new['StartDate'] = pd.to_datetime(EMA_data_new['StartDate'], format="%m/%d/%Y %H:%M")
Intake_data_new = pd.read_csv(Data_path2)

# Scratch
output_folder = "Z:\Projects\EMA_Project\Scripts\Output\Multiverse_Scratch"
os.makedirs(output_folder, exist_ok=True)

## Filter and Save

In [ ]:
for filter in filtering:
    # Filtering logic
    if filter == "More_Than_10_Responses":
        EMA_data_temp = EMA_data_new.groupby('tempid').filter(lambda x: x['posAff'].notna().sum() >= 10)
    elif filter == "More_Than_20_Seconds":
        EMA_data_temp = EMA_data_new[EMA_data_new['duration'] >= 20]
    elif filter == "MT10R_AND_MT20S":
        EMA_data_temp = EMA_data_new[EMA_data_new['duration'] >= 20]
        EMA_data_temp = EMA_data_temp.groupby('tempid').filter(lambda x: x['posAff'].notna().sum() >= 10)
    elif filter == "No_Filtering":
        EMA_data_temp = EMA_data_new

    # Steps regardless of filtering
    EMA_data_temp = EMA_data_temp.sort_values(by=['tempid', 'StartDate'])
    EMA_data_temp['time'] = EMA_data_temp.groupby('tempid').cumcount() + 1
    EMA_data_filtered = EMA_data_temp
    Intake_data_temp = Intake_data_new

    # Filter participants with both EMA and Intake data
    valid_ids = EMA_data_filtered['tempid'].unique()
    valid_ids = set(valid_ids).intersection(Intake_data_temp['tempid'].unique())

    EMA_data_filtered = EMA_data_filtered[EMA_data_filtered['tempid'].isin(valid_ids)]
    Intake_data_temp = Intake_data_temp[Intake_data_temp['tempid'].isin(valid_ids)]

    # Create Time Series
    posAffTS = reshape_data(EMA_data_filtered, 'posAff')
    negAffTS = reshape_data(EMA_data_filtered, 'negAff')
    stressTS = reshape_data(EMA_data_filtered, 'stress')
    selfER_1TS = reshape_data(EMA_data_filtered, 'selfER_1')  # Reappraisal
    selfER_2TS = reshape_data(EMA_data_filtered, 'selfER_2')  # Suppression
    selfER_3TS = reshape_data(EMA_data_filtered, 'selfER_3')  # Distraction
    selfER_4TS = reshape_data(EMA_data_filtered, 'selfER_4')  # Selective Attention
    selfER_5TS = reshape_data(EMA_data_filtered, 'selfER_5')  # Situation Selection

    # Save outputs
    EMA_data_filtered.to_csv(os.path.join(output_folder, f"EMA_data_filtered_{filter}.csv"), index=False)
    Intake_data_temp.to_csv(os.path.join(output_folder, f"Intake_data_temp_{filter}.csv"), index=False)
    posAffTS.to_csv(os.path.join(output_folder, f"posAffTS_{filter}.csv"))
    negAffTS.to_csv(os.path.join(output_folder, f"negAffTS_{filter}.csv"))
    stressTS.to_csv(os.path.join(output_folder, f"stressTS_{filter}.csv"))
    selfER_1TS.to_csv(os.path.join(output_folder, f"selfER_1TS_{filter}.csv"))
    selfER_2TS.to_csv(os.path.join(output_folder, f"selfER_2TS_{filter}.csv"))
    selfER_3TS.to_csv(os.path.join(output_folder, f"selfER_3TS_{filter}.csv"))
    selfER_4TS.to_csv(os.path.join(output_folder, f"selfER_4TS_{filter}.csv"))
    selfER_5TS.to_csv(os.path.join(output_folder, f"selfER_5TS_{filter}.csv"))

print("Filtering saved")

## Create Transition Matrix Lists and Save

In [ ]:
for filter in filtering:
    # Load requisite files for the current filter
    posAffTS = pd.read_csv(os.path.join(output_folder, f"posAffTS_{filter}.csv"), index_col=0)
    negAffTS = pd.read_csv(os.path.join(output_folder, f"negAffTS_{filter}.csv"), index_col=0)
    stressTS = pd.read_csv(os.path.join(output_folder, f"stressTS_{filter}.csv"), index_col=0)
    selfER_1TS = pd.read_csv(os.path.join(output_folder, f"selfER_1TS_{filter}.csv"), index_col=0)
    selfER_2TS = pd.read_csv(os.path.join(output_folder, f"selfER_2TS_{filter}.csv"), index_col=0)
    selfER_3TS = pd.read_csv(os.path.join(output_folder, f"selfER_3TS_{filter}.csv"), index_col=0)
    selfER_4TS = pd.read_csv(os.path.join(output_folder, f"selfER_4TS_{filter}.csv"), index_col=0)
    selfER_5TS = pd.read_csv(os.path.join(output_folder, f"selfER_5TS_{filter}.csv"), index_col=0)

    for definition in definition_transition_matrix:
        if definition == "Transition":
            PA_timeseries_list, PA_transition_matrices = indiv_ts_tmat(posAffTS)
            NA_timeseries_list, NA_transition_matrices = indiv_ts_tmat(negAffTS)
            ST_timeseries_list, ST_transition_matrices = indiv_ts_tmat(stressTS)
            RP_timeseries_list, RP_transition_matrices = indiv_ts_tmat(selfER_1TS)
            SP_timeseries_list, SP_transition_matrices = indiv_ts_tmat(selfER_2TS)
            DS_timeseries_list, DS_transition_matrices = indiv_ts_tmat(selfER_3TS)
            SA_timeseries_list, SA_transition_matrices = indiv_ts_tmat(selfER_4TS)
            SS_timeseries_list, SS_transition_matrices = indiv_ts_tmat(selfER_5TS)
        elif definition == "Proportion":
            PA_timeseries_list, PA_transition_matrices = indiv_ts_pmat(posAffTS)
            NA_timeseries_list, NA_transition_matrices = indiv_ts_pmat(negAffTS)
            ST_timeseries_list, ST_transition_matrices = indiv_ts_pmat(stressTS)
            RP_timeseries_list, RP_transition_matrices = indiv_ts_pmat(selfER_1TS)
            SP_timeseries_list, SP_transition_matrices = indiv_ts_pmat(selfER_2TS)
            DS_timeseries_list, DS_transition_matrices = indiv_ts_pmat(selfER_3TS)
            SA_timeseries_list, SA_transition_matrices = indiv_ts_pmat(selfER_4TS)
            SS_timeseries_list, SS_transition_matrices = indiv_ts_pmat(selfER_5TS)

        # Convert transition_matrices (list of ndarrays) to DataFrames and save
        for name, matrices in zip(["PA", "NA", "ST", "RP", "SP", "DS", "SA", "SS"], 
                          [PA_transition_matrices, NA_transition_matrices, ST_transition_matrices, 
                           RP_transition_matrices, SP_transition_matrices, DS_transition_matrices, 
                           SA_transition_matrices, SS_transition_matrices]):
            # Flatten each matrix to 1D and then stack them
            flattened_matrices = [matrix.ravel() for matrix in matrices]  # Flatten each 5x5 to a 1D array of 25
            matrices_df = pd.DataFrame(flattened_matrices)  # Create a DataFrame with 25 columns
            matrices_df.to_csv(os.path.join(output_folder, f"{name}_transition_matrices_{filter}_{definition}.csv"), index=False)

        # Convert timeseries_list (list of lists) to DataFrames and save
        for name, timeseries in zip(["PA", "NA", "ST", "RP", "SP", "DS", "SA", "SS"], 
                                    [PA_timeseries_list, NA_timeseries_list, ST_timeseries_list, 
                                     RP_timeseries_list, SP_timeseries_list, DS_timeseries_list, 
                                     SA_timeseries_list, SS_timeseries_list]):
            timeseries_df = pd.DataFrame(timeseries)
            timeseries_df.to_csv(os.path.join(output_folder, f"{name}_timeseries_list_{filter}_{definition}.csv"), index=False)

print("Matrix definition completed.")

## Create all RDMs (and LOSO) and Save

In [ ]:
for filter in filtering:
    print(f"Processing filter: {filter}")
    for definition in definition_transition_matrix:
        print(f"Processing definition: {definition}")
        try:
            # Load the transition matrices from the intermediate results
            print("Loading transition matrices...")
            PA_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"PA_transition_matrices_{filter}_{definition}.csv")).values
            NA_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"NA_transition_matrices_{filter}_{definition}.csv")).values
            ST_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"ST_transition_matrices_{filter}_{definition}.csv")).values
            RP_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"RP_transition_matrices_{filter}_{definition}.csv")).values
            SP_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"SP_transition_matrices_{filter}_{definition}.csv")).values
            DS_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"DS_transition_matrices_{filter}_{definition}.csv")).values
            SA_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"SA_transition_matrices_{filter}_{definition}.csv")).values
            SS_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"SS_transition_matrices_{filter}_{definition}.csv")).values

            print("Reshaping transition matrices...")
            PA_transition_matrices = [matrix.reshape(5, 5) for matrix in PA_transition_matrices_flat]
            NA_transition_matrices = [matrix.reshape(5, 5) for matrix in NA_transition_matrices_flat]
            ST_transition_matrices = [matrix.reshape(5, 5) for matrix in ST_transition_matrices_flat]
            RP_transition_matrices = [matrix.reshape(5, 5) for matrix in RP_transition_matrices_flat]
            SP_transition_matrices = [matrix.reshape(5, 5) for matrix in SP_transition_matrices_flat]
            DS_transition_matrices = [matrix.reshape(5, 5) for matrix in DS_transition_matrices_flat]
            SA_transition_matrices = [matrix.reshape(5, 5) for matrix in SA_transition_matrices_flat]
            SS_transition_matrices = [matrix.reshape(5, 5) for matrix in SS_transition_matrices_flat]

            for measure in distance_measure:
                print(f"Processing measure: {measure}")
                # Get measure in function format
                if measure == "Spearman":
                    use = "spear"
                elif measure == "Pearson":
                    use = "pearson"
                elif measure == "Euclid":
                    use = "euclid"

                for comparison in comparison_method:
                    print(f"Processing comparison: {comparison}")
                    # Get comparison in function format
                    if comparison == "Leave-One-Subject-Out":
                        comp = "loso_similarity_matrix"
                    elif comparison == "Pairwise":
                        comp = "compute_rsm"

                    function_name = f"{comp}_{use}"
                    print(f"Calling function: {function_name}")
                    
                    # Calculate similarity matrices
                    PArsm = globals()[function_name](PA_transition_matrices)
                    NArsm = globals()[function_name](NA_transition_matrices)
                    STrsm = globals()[function_name](ST_transition_matrices)
                    RPrsm = globals()[function_name](RP_transition_matrices)
                    SPrsm = globals()[function_name](SP_transition_matrices)
                    DSrsm = globals()[function_name](DS_transition_matrices)
                    SArsm = globals()[function_name](SA_transition_matrices)
                    SSrsm = globals()[function_name](SS_transition_matrices)

                    # Make into dissimilarity (and take lower triangle if necessary)
                    print("Calculating dissimilarity matrices...")
                    if comparison == "Leave-One-Subject-Out":
                        if measure == "Pearson" or measure == "Spearman":
                            PArdm = [1 - corr for corr in PArsm]
                            NArdm = [1 - corr for corr in NArsm]
                            STrdm = [1 - corr for corr in STrsm]
                            RPrdm = [1 - corr for corr in RPrsm]
                            SPrdm = [1 - corr for corr in SPrsm]
                            DSrdm = [1 - corr for corr in DSrsm]
                            SArdm = [1 - corr for corr in SArsm]
                            SSrdm = [1 - corr for corr in SSrsm]
                        elif measure == "Euclid":
                            PArdm = [2 * corr for corr in PArsm] # scale to match Pearson and Spearman
                            NArdm = [2 * corr for corr in NArsm]
                            STrdm = [2 * corr for corr in STrsm]
                            RPrdm = [2 * corr for corr in RPrsm]
                            SPrdm = [2 * corr for corr in SPrsm]
                            DSrdm = [2 * corr for corr in DSrsm]
                            SArdm = [2 * corr for corr in SArsm]
                            SSrdm = [2 * corr for corr in SSrsm]
                        
                        # Identify non-NaN indices for PArdm as a reference
                        PArdm = np.array(PArdm)
                        valid_indices = np.where(~np.isnan(PArdm))[0]
                        np.save(os.path.join(output_folder, f"valid_indices_{filter}_{definition}_{measure}_{comparison}.npy"), valid_indices)
                        
                        # Apply valid indices
                        PArdm = PArdm[valid_indices]
                        NArdm = np.array(NArdm)[valid_indices]
                        STrdm = np.array(STrdm)[valid_indices]
                        RPrdm = np.array(RPrdm)[valid_indices]
                        SPrdm = np.array(SPrdm)[valid_indices]
                        DSrdm = np.array(DSrdm)[valid_indices]
                        SArdm = np.array(SArdm)[valid_indices]
                        SSrdm = np.array(SSrdm)[valid_indices]
                        
                    elif comparison == "Pairwise":
                        if measure == "Pearson" or measure == "Spearman":
                            PArdm = 1 - PArsm
                            NArdm = 1 - NArsm
                            STrdm = 1 - STrsm
                            RPrdm = 1 - RPrsm
                            SPrdm = 1 - SPrsm
                            DSrdm = 1 - DSrsm
                            SArdm = 1 - SArsm
                            SSrdm = 1 - SSrsm
                        elif measure == "Euclid":
                            PArdm = 2 * PArsm
                            NArdm = 2 * NArsm
                            STrdm = 2 * STrsm
                            RPrdm = 2 * RPrsm
                            SPrdm = 2 * SPrsm
                            DSrdm = 2 * DSrsm
                            SArdm = 2 * SArsm
                            SSrdm = 2 * SSrsm
                            
                        # Remove NaN values and save valid indices
                        print("Removing NaN values...")
                        PArdm = np.array(PArdm)
                        lower_triangle_indices = np.tril_indices_from(PArdm, k=-1)
                        PArdm = PArdm[lower_triangle_indices]
                        NArdm = NArdm[lower_triangle_indices]
                        STrdm = STrdm[lower_triangle_indices]
                        RPrdm = RPrdm[lower_triangle_indices]
                        SPrdm = SPrdm[lower_triangle_indices]
                        DSrdm = DSrdm[lower_triangle_indices]
                        SArdm = SArdm[lower_triangle_indices]
                        SSrdm = SSrdm[lower_triangle_indices]
                        
                        valid_mask = ~np.isnan(PArdm)
                        valid_indices = np.where(valid_mask)[0]
                        PArdm = PArdm[valid_indices]
                        NArdm = NArdm[valid_indices]
                        STrdm = STrdm[valid_indices]
                        RPrdm = RPrdm[valid_indices]
                        SPrdm = SPrdm[valid_indices]
                        DSrdm = DSrdm[valid_indices]
                        SArdm = SArdm[valid_indices]
                        SSrdm = SSrdm[valid_indices]
                        np.save(os.path.join(output_folder, f"valid_indices_{filter}_{definition}_{measure}_{comparison}.npy"), valid_indices)
                        np.save(os.path.join(output_folder, f"lower_triangle_indices_{filter}_{definition}_{measure}_{comparison}.npy"), lower_triangle_indices)


                    # Save the dissimilarity matrices
                    print("Saving results...")
                    # List of matrix names and corresponding arrays
                    matrix_names = ["PArdm", "NArdm", "STrdm", "RPrdm", "SPrdm", "DSrdm", "SArdm", "SSrdm"]
                    matrices = [PArdm, NArdm, STrdm, RPrdm, SPrdm, DSrdm, SArdm, SSrdm]
                    
                    for name, matrix in zip(matrix_names, matrices):
                        # Reshape the array to (XXX, 1) to ensure proper saving
                        matrix = matrix.reshape(-1, 1)
                        file_path = os.path.join(output_folder, f"{name}_{filter}_{definition}_{measure}_{comparison}.npy")
                        np.save(file_path, matrix)
                        
                    print("Iteration completed.")
                    time.sleep(10)

            # Clean up memory
            del PA_transition_matrices, NA_transition_matrices, ST_transition_matrices, RP_transition_matrices
            del SP_transition_matrices, DS_transition_matrices, SA_transition_matrices, SS_transition_matrices
            gc.collect()

        except Exception as e:
            print(f"Error encountered: {e}")
            continue

print("Distance comparison completed.")

In [ ]:
# HELPER FUNCTIONS FOR MATCHING COVARIATES TO VARIABLES OF INTEREST
def load_lower_triangle_indices(filter, definition, measure, comparison):
    file_path = os.path.join(output_folder, f"lower_triangle_indices_{filter}_{definition}_{measure}_{comparison}.npy")
    if os.path.exists(file_path):
        return np.load(file_path, allow_pickle=True)
    else:
        return None

def load_valid_indices(filter, definition, measure, comparison):
    file_path = os.path.join(output_folder, f"valid_indices_{filter}_{definition}_{measure}_{comparison}.npy")
    if os.path.exists(file_path):
        return np.load(file_path, allow_pickle=True)
    else:
        return None

## Save Time Covariates

In [ ]:
for filter in filtering:
    # Grab the right EMA filtered data and set it equal to EMA_data_temp2
    EMA_data_temp2 = pd.read_csv(os.path.join(output_folder, f"EMA_data_filtered_{filter}.csv"))

    # Make sure it's in datetime format (can be messed up by csv saving)
    if not pd.api.types.is_datetime64_any_dtype(EMA_data_temp2['StartDate']):
        EMA_data_temp2['StartDate'] = pd.to_datetime(EMA_data_temp2['StartDate'])

    # Extract date from StartDate
    EMA_data_temp2['Day'] = pd.to_datetime(EMA_data_temp2['StartDate']).dt.date

    # Calculate LOSO start date
    start_days = EMA_data_temp2.groupby('tempid')['Day'].min().reset_index()
    start_days_l = start_days['Day'].tolist()

    # Save
    for definition in definition_transition_matrix:
        for measure in distance_measure:
            for comparison in comparison_method:
                lower_triangle_indices = load_lower_triangle_indices(filter, definition, measure, comparison)
                valid_indices = load_valid_indices(filter, definition, measure, comparison)
                if lower_triangle_indices is None:
                    startDate_loso = calculate_date_differences_loso(start_days_l)
                    startDate_loso = np.array(startDate_loso)[valid_indices]
                    np.save(os.path.join(output_folder, f"startDate_{filter}_{definition}_{measure}_{comparison}.npy"), startDate_loso)
                if lower_triangle_indices is not None:
                    lower_triangle_indices = tuple(lower_triangle_indices)
                    normalized_date_diff_matrix = calculate_date_differences(start_days_l)
                    startDate = np.array(normalized_date_diff_matrix)[lower_triangle_indices]
                    startDate = startDate[valid_indices]
                    np.save(os.path.join(output_folder, f"startDate_{filter}_{definition}_{measure}_{comparison}.npy"), startDate)

    # Save
    for definition in definition_transition_matrix:
        for measure in distance_measure:
            for comparison in comparison_method:
                lower_triangle_indices = load_lower_triangle_indices(filter, definition, measure, comparison)
                valid_indices = load_valid_indices(filter, definition, measure, comparison)
                if lower_triangle_indices is None:
                    responseWindowLoso = calculate_response_windows_loso(EMA_data_temp2)
                    responseWindowLoso = np.array(responseWindowLoso)[valid_indices]
                    np.save(os.path.join(output_folder, f"responseWindow_{filter}_{definition}_{measure}_{comparison}.npy"), responseWindowLoso)
                if lower_triangle_indices is not None:
                    lower_triangle_indices = tuple(lower_triangle_indices)
                    response_window_matrix = calculate_response_windows(EMA_data_temp2)
                    responseWindow = np.array(response_window_matrix)[lower_triangle_indices]
                    responseWindow = responseWindow[valid_indices]
                    np.save(os.path.join(output_folder, f"responseWindow_{filter}_{definition}_{measure}_{comparison}.npy"), responseWindow)

## Save Demographic Covariate

In [ ]:
# CONTROL FOR DEMOGRAPHICS
for filter in filtering:
    # Grab the right intake filtered data and set it equal to Intake_data_temp
    Intake_data_temp = pd.read_csv(os.path.join(output_folder, f"Intake_data_temp_{filter}.csv"))

    # fix demographic NAs
    race_vars = ['race_1_black', 'race_2_asian', 'race_3_nathi_pacisl',
                  'race_4_white', 'race_5_amin_alnat', 'race_6_other']
    Intake_data_temp[race_vars] = Intake_data_temp[race_vars].fillna(0).astype(int)
    
    
    Demographic_variables = ['sex', 'gender', 'eth', 'race_1_black',
                             'race_2_asian', 'race_3_nathi_pacisl',	
                             'race_4_white', 'race_5_amin_alnat', 
                             'race_6_other', 'par_fin_help']

    for definition in definition_transition_matrix:
        for measure in distance_measure:
            for comparison in comparison_method:
                lower_triangle_indices = load_lower_triangle_indices(filter, definition, measure, comparison)
                valid_indices = load_valid_indices(filter, definition, measure, comparison)
                if comparison == "Leave-One-Subject-Out":
                    demographics_loso = loso_demo_euclidean(Intake_data_temp, Demographic_variables)
                    demographics_loso = demographics_loso[valid_indices]
                    np.save(os.path.join(output_folder, f"demo_{filter}_{definition}_{measure}_{comparison}.npy"), demographics_loso)
                elif comparison == "Pairwise":
                    lower_triangle_indices = tuple(lower_triangle_indices)
                    demographics_pairwise = rsm_demo_euclidean(Intake_data_temp, Demographic_variables)
                    demographics_pairwise =  demographics_pairwise[lower_triangle_indices]
                    demographics_pairwise =  demographics_pairwise[valid_indices]
                    np.save(os.path.join(output_folder, f"demo_{filter}_{definition}_{measure}_{comparison}.npy"), demographics_pairwise)

## Save Baseline Emotion Regulation Use Covariate

In [ ]:
# Map the key to the subscale columns
erq_mapping = {
    'RP': ['ERQ_RP_01', 'ERQ_RP_02', 'ERQ_RP_03', 'ERQ_RP_04', 'ERQ_RP_05', 'ERQ_RP_06'],
    'DS': ['ERQ_DS_01', 'ERQ_DS_02', 'ERQ_DS_03', 'ERQ_DS_04', 'ERQ_DS_05'],
    'SP': ['ERQ_SP_01', 'ERQ_SP_02', 'ERQ_SP_03', 'ERQ_SP_04'],
    'SA': ['ERQ_SA_01', 'ERQ_SA_02', 'ERQ_SA_03', 'ERQ_SA_04'],
    'SS': ['ERQ_SS_01', 'ERQ_SS_02', 'ERQ_SS_03']
}

# List of variables
EERQ_variables = list(erq_mapping.keys())

for filter in filtering:
    Intake_data_temp = pd.read_csv(os.path.join(output_folder, f"Intake_data_temp_{filter}.csv"))

    # Calculate subscale scores based on the mapping
    for subscale, columns in erq_mapping.items():
        Intake_data_temp[subscale] = Intake_data_temp[columns].sum(axis=1)

    for definition in definition_transition_matrix:
        for measure in distance_measure:
            for comparison in comparison_method:
                lower_triangle_indices = load_lower_triangle_indices(filter, definition, measure, comparison)
                valid_indices = load_valid_indices(filter, definition, measure, comparison)
                if comparison == "Leave-One-Subject-Out":
                    EERQ_loso = loso_strat_euclidean(Intake_data_temp, EERQ_variables)
                    EERQ_loso = EERQ_loso[valid_indices]
                    np.save(os.path.join(output_folder, f"EERQ_{filter}_{definition}_{measure}_{comparison}.npy"), EERQ_loso)
                elif comparison == "Pairwise":
                    lower_triangle_indices = tuple(lower_triangle_indices)
                    EERQ_pairwise = rsm_strat_euclidean(Intake_data_temp, EERQ_variables)
                    EERQ_pairwise = EERQ_pairwise[lower_triangle_indices]
                    EERQ_pairwise = EERQ_pairwise[valid_indices]
                    np.save(os.path.join(output_folder, f"EERQ_{filter}_{definition}_{measure}_{comparison}.npy"), EERQ_pairwise)

## Save Entropy Covariate

In [ ]:
for filter in filtering:
    for definition in definition_transition_matrix:
        # Load the transition matrices from the intermediate results
        PA_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"PA_transition_matrices_{filter}_{definition}.csv")).values
        NA_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"NA_transition_matrices_{filter}_{definition}.csv")).values
        ST_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"ST_transition_matrices_{filter}_{definition}.csv")).values
        RP_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"RP_transition_matrices_{filter}_{definition}.csv")).values
        SP_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"SP_transition_matrices_{filter}_{definition}.csv")).values
        DS_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"DS_transition_matrices_{filter}_{definition}.csv")).values
        SA_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"SA_transition_matrices_{filter}_{definition}.csv")).values
        SS_transition_matrices_flat = pd.read_csv(os.path.join(output_folder, f"SS_transition_matrices_{filter}_{definition}.csv")).values

        # Get back into original format
        PA_transition_matrices = [matrix.reshape(5, 5) for matrix in PA_transition_matrices_flat]
        NA_transition_matrices = [matrix.reshape(5, 5) for matrix in NA_transition_matrices_flat]
        ST_transition_matrices = [matrix.reshape(5, 5) for matrix in ST_transition_matrices_flat]
        RP_transition_matrices = [matrix.reshape(5, 5) for matrix in RP_transition_matrices_flat]
        SP_transition_matrices = [matrix.reshape(5, 5) for matrix in SP_transition_matrices_flat]
        DS_transition_matrices = [matrix.reshape(5, 5) for matrix in DS_transition_matrices_flat]
        SA_transition_matrices = [matrix.reshape(5, 5) for matrix in SA_transition_matrices_flat]
        SS_transition_matrices = [matrix.reshape(5, 5) for matrix in SS_transition_matrices_flat]
        
        for measure in distance_measure:
            for comparison in comparison_method:
                if comparison == "Leave-One-Subject-Out":
                    PAentropy = calculate_entropy_per_subject(PA_transition_matrices)
                    NAentropy = calculate_entropy_per_subject(NA_transition_matrices)
                    STentropy = calculate_entropy_per_subject(ST_transition_matrices)
                    RPentropy = calculate_entropy_per_subject(RP_transition_matrices)
                    SPentropy = calculate_entropy_per_subject(SP_transition_matrices)
                    DSentropy = calculate_entropy_per_subject(DS_transition_matrices)
                    SAentropy = calculate_entropy_per_subject(SA_transition_matrices)
                    SSentropy = calculate_entropy_per_subject(SS_transition_matrices)
                elif comparison == "Pairwise":
                    PAentropyMat = calculate_entropy_differences(PA_transition_matrices)
                    NAentropyMat = calculate_entropy_differences(NA_transition_matrices)
                    STentropyMat = calculate_entropy_differences(ST_transition_matrices)
                    RPentropyMat = calculate_entropy_differences(RP_transition_matrices)
                    SPentropyMat = calculate_entropy_differences(SP_transition_matrices)
                    DSentropyMat = calculate_entropy_differences(DS_transition_matrices)
                    SAentropyMat = calculate_entropy_differences(SA_transition_matrices)
                    SSentropyMat = calculate_entropy_differences(SS_transition_matrices)
    
                # Save
                lower_triangle_indices = load_lower_triangle_indices(filter, definition, measure, comparison)
                valid_indices = load_valid_indices(filter, definition, measure, comparison)
                if comparison == "Leave-One-Subject-Out":
                    PAentropy = np.array(PAentropy)[valid_indices]
                    NAentropy = np.array(NAentropy)[valid_indices]
                    STentropy = np.array(STentropy)[valid_indices]
                    RPentropy = np.array(RPentropy)[valid_indices]
                    SPentropy = np.array(SPentropy)[valid_indices]
                    DSentropy = np.array(DSentropy)[valid_indices]
                    SAentropy = np.array(SAentropy)[valid_indices]
                    SSentropy = np.array(SSentropy)[valid_indices]
                    np.save(os.path.join(output_folder, f"PAentropy_{filter}_{definition}_{measure}_{comparison}.npy"), PAentropy)
                    np.save(os.path.join(output_folder, f"NAentropy_{filter}_{definition}_{measure}_{comparison}.npy"), NAentropy)
                    np.save(os.path.join(output_folder, f"STentropy_{filter}_{definition}_{measure}_{comparison}.npy"), STentropy)
                    np.save(os.path.join(output_folder, f"RPentropy_{filter}_{definition}_{measure}_{comparison}.npy"), RPentropy)
                    np.save(os.path.join(output_folder, f"SPentropy_{filter}_{definition}_{measure}_{comparison}.npy"), SPentropy)
                    np.save(os.path.join(output_folder, f"DSentropy_{filter}_{definition}_{measure}_{comparison}.npy"), DSentropy)
                    np.save(os.path.join(output_folder, f"SAentropy_{filter}_{definition}_{measure}_{comparison}.npy"), SAentropy)
                    np.save(os.path.join(output_folder, f"SSentropy_{filter}_{definition}_{measure}_{comparison}.npy"), SSentropy)
                elif comparison == "Pairwise":
                    lower_triangle_indices = tuple(lower_triangle_indices)
                    PAentropy = PAentropyMat[lower_triangle_indices]
                    NAentropy = NAentropyMat[lower_triangle_indices]
                    STentropy = STentropyMat[lower_triangle_indices]
                    RPentropy = RPentropyMat[lower_triangle_indices]
                    SPentropy = SPentropyMat[lower_triangle_indices]
                    DSentropy = DSentropyMat[lower_triangle_indices]
                    SAentropy = SAentropyMat[lower_triangle_indices]
                    SSentropy = SSentropyMat[lower_triangle_indices]
                    PAentropy = PAentropy[valid_indices]
                    NAentropy = NAentropy[valid_indices]
                    STentropy = STentropy[valid_indices]
                    RPentropy = RPentropy[valid_indices]
                    SPentropy = SPentropy[valid_indices]
                    DSentropy = DSentropy[valid_indices]
                    SAentropy = SAentropy[valid_indices]
                    SSentropy = SSentropy[valid_indices]
                    np.save(os.path.join(output_folder, f"PAentropy_{filter}_{definition}_{measure}_{comparison}.npy"), PAentropy)
                    np.save(os.path.join(output_folder, f"NAentropy_{filter}_{definition}_{measure}_{comparison}.npy"), NAentropy)
                    np.save(os.path.join(output_folder, f"STentropy_{filter}_{definition}_{measure}_{comparison}.npy"), STentropy)
                    np.save(os.path.join(output_folder, f"RPentropy_{filter}_{definition}_{measure}_{comparison}.npy"), RPentropy)
                    np.save(os.path.join(output_folder, f"SPentropy_{filter}_{definition}_{measure}_{comparison}.npy"), SPentropy)
                    np.save(os.path.join(output_folder, f"DSentropy_{filter}_{definition}_{measure}_{comparison}.npy"), DSentropy)
                    np.save(os.path.join(output_folder, f"SAentropy_{filter}_{definition}_{measure}_{comparison}.npy"), SAentropy)
                    np.save(os.path.join(output_folder, f"SSentropy_{filter}_{definition}_{measure}_{comparison}.npy"), SSentropy)

## RUN REGRESSIONS AND SAVE RESULTS

In [13]:
# Directory to save the results
results_folder = "Z:\Projects\EMA_Project\Scripts\Output\Multiverse_Results"

# Ensure the folder exists
os.makedirs(results_folder, exist_ok=True)

<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
C:\Users\CDN Lab\AppData\Local\Temp\ipykernel_25292\3199816356.py:2: SyntaxWarning: invalid escape sequence '\P'
  results_folder = "Z:\Projects\EMA_Project\Scripts\Output\Multiverse_Results"


In [14]:
# Helper function for this part, because some RDMs seem to be coming in as X, 1
def squeeze_array(arr):
    """
    Ensure array has shape (X,) instead of (X, 1)
    """
    if arr.ndim == 2 and arr.shape[1] == 1:
        return arr.squeeze(axis=1)
    return arr.squeeze() if arr.ndim > 1 and arr.shape[-1] == 1 else arr

In [ ]:
for filter in filtering:
    for definition in definition_transition_matrix:
        for measure in distance_measure:
            for comparison in comparison_method:
                # Load the RDMs from the intermediate results
                PArdm = squeeze_array(np.load(os.path.join(output_folder, f"PArdm_{filter}_{definition}_{measure}_{comparison}.npy")))
                NArdm = squeeze_array(np.load(os.path.join(output_folder, f"NArdm_{filter}_{definition}_{measure}_{comparison}.npy")))
                STrdm = squeeze_array(np.load(os.path.join(output_folder, f"STrdm_{filter}_{definition}_{measure}_{comparison}.npy")))
                RPrdm = squeeze_array(np.load(os.path.join(output_folder, f"RPrdm_{filter}_{definition}_{measure}_{comparison}.npy")))
                SPrdm = squeeze_array(np.load(os.path.join(output_folder, f"SPrdm_{filter}_{definition}_{measure}_{comparison}.npy")))
                DSrdm = squeeze_array(np.load(os.path.join(output_folder, f"DSrdm_{filter}_{definition}_{measure}_{comparison}.npy")))
                SArdm = squeeze_array(np.load(os.path.join(output_folder, f"SArdm_{filter}_{definition}_{measure}_{comparison}.npy")))
                SSrdm = squeeze_array(np.load(os.path.join(output_folder, f"SSrdm_{filter}_{definition}_{measure}_{comparison}.npy")))
                
                # Load all covariates
                # ENTROPY
                PAentropy = squeeze_array(np.load(os.path.join(output_folder, f"PAentropy_{filter}_{definition}_{measure}_{comparison}.npy")))
                NAentropy = squeeze_array(np.load(os.path.join(output_folder, f"NAentropy_{filter}_{definition}_{measure}_{comparison}.npy")))
                STentropy = squeeze_array(np.load(os.path.join(output_folder, f"STentropy_{filter}_{definition}_{measure}_{comparison}.npy")))
                RPentropy = squeeze_array(np.load(os.path.join(output_folder, f"RPentropy_{filter}_{definition}_{measure}_{comparison}.npy")))
                SPentropy = squeeze_array(np.load(os.path.join(output_folder, f"SPentropy_{filter}_{definition}_{measure}_{comparison}.npy")))
                DSentropy = squeeze_array(np.load(os.path.join(output_folder, f"DSentropy_{filter}_{definition}_{measure}_{comparison}.npy")))
                SAentropy = squeeze_array(np.load(os.path.join(output_folder, f"SAentropy_{filter}_{definition}_{measure}_{comparison}.npy")))
                SSentropy = squeeze_array(np.load(os.path.join(output_folder, f"SSentropy_{filter}_{definition}_{measure}_{comparison}.npy")))

                # DEMO
                demographics = squeeze_array(np.load(os.path.join(output_folder, f"demo_{filter}_{definition}_{measure}_{comparison}.npy")))
                
                # TIME
                startDate = squeeze_array(np.load(os.path.join(output_folder, f"startDate_{filter}_{definition}_{measure}_{comparison}.npy")))
                responseWindow = squeeze_array(np.load(os.path.join(output_folder, f"responseWindow_{filter}_{definition}_{measure}_{comparison}.npy")))
                
                # EERQ
                EERQ = squeeze_array(np.load(os.path.join(output_folder, f"EERQ_{filter}_{definition}_{measure}_{comparison}.npy")))
                                
                for entropy in control_for_entropy:
                    for demo in control_for_demo:
                        for time in temporal_overlap:
                            for ER in baseline_ERS_use:
                                # Fresh version of covariates and corresponding labels
                                core_covariates = [RPrdm, SPrdm, DSrdm, SArdm, SSrdm]
                                core_labels = ["RP", "SP", "DS", "SA", "SS"]
                                outcomes = {"PA": PArdm, "NA": NArdm, "ST": STrdm}

                                optional_covariates = []
                                optional_labels = []
                                if time == "Start_Date" or time == "Date_and_Time":
                                    optional_covariates.append(startDate)
                                    optional_labels.append("startDate")
                                
                                if time == "Response_Window" or time == "Date_and_Time":
                                    optional_covariates.append(responseWindow)
                                    optional_labels.append("responseWindow")
                                
                                if demo == "Demographics":
                                    optional_covariates.append(demographics)
                                    optional_labels.append("demographics")
                                
                                if ER == "Baseline_ER":
                                    optional_covariates.append(EERQ)
                                    optional_labels.append("EERQ")
                                    
                                entropy_covariates = {
                                    "PA": [PAentropy, RPentropy, SPentropy, DSentropy, SAentropy, SSentropy],
                                    "NA": [NAentropy, RPentropy, SPentropy, DSentropy, SAentropy, SSentropy],
                                    "ST": [STentropy, RPentropy, SPentropy, DSentropy, SAentropy, SSentropy],
                                }
                                entropy_labels = ["PAentropy", "RPentropy", "SPentropy", "DSentropy", "SAentropy", "SSentropy"]
                            
                                results_list = {"PA": [], "NA": [], "ST": []}
                                analytic_dimensions = []
                                
                                for outcome_label, outcome_data in outcomes.items():
                                    X = core_covariates + optional_covariates[:]
                                    labels = core_labels + optional_labels[:]
                                
                                    if entropy == "Entropy":
                                        X += entropy_covariates[outcome_label]
                                        labels += entropy_labels
                                
                                    X_mat = sm.add_constant(np.column_stack(X))
                                    variable_labels = {f"x{i+1}": lbl for i, lbl in enumerate(labels)}
                                    variable_labels = {'const': 'Intercept', **variable_labels}
                                
                                    model = sm.OLS(outcome_data, X_mat)
                                    results = model.fit()
                                
                                    param_names = results.model.exog_names
                                    df = pd.DataFrame({
                                        "Variable": [variable_labels.get(name, name) for name in param_names],
                                        "Parameter": results.params,
                                        "Std_Err": results.bse,
                                        "P_value": results.pvalues,
                                        "Observations": [results.nobs] * len(results.params)
                                    })

                                    for label in core_labels:
                                        specification = f"{filter}_{definition}_{measure}_{comparison}_{entropy}_{demo}_{time}_{ER}"
                                        filename = f"{outcome_label}_{label}_result_{specification}.csv"
                                        df_label = df[df["Variable"] == label]
                                        if not df_label.empty:
                                            df_label.to_csv(os.path.join(results_folder, filename), index=False)
                                
                                    results_list[outcome_label].append(results)
                                    analytic_dimensions.append(specification)

                                # Single-Predictor Regressions
                                for outcome_label, outcome_data in outcomes.items():
                                    for predictor_data, predictor_label in zip(core_covariates, core_labels):
                                        # Start with the single core predictor
                                        X_single = [predictor_data]
                                        labels_single = [predictor_label]

                                        # Add optional covariates
                                        X_single += optional_covariates[:]
                                        labels_single += optional_labels[:]

                                        # Add entropy covariates if applicable
                                        if entropy == "Entropy":
                                            X_single += entropy_covariates[outcome_label]
                                            labels_single += entropy_labels

                                        # Stack into design matrix
                                        X_single_mat = sm.add_constant(np.column_stack(X_single))
                                        variable_labels_single = {f"x{i+1}": lbl for i, lbl in enumerate(labels_single)}
                                        variable_labels_single = {'const': 'Intercept', **variable_labels_single}

                                        model_single = sm.OLS(outcome_data, X_single_mat)
                                        results_single = model_single.fit()

                                        param_names_single = results_single.model.exog_names
                                        df_single = pd.DataFrame({
                                            "Variable": [variable_labels_single.get(name, name) for name in param_names_single],
                                            "Parameter": results_single.params,
                                            "Std_Err": results_single.bse,
                                            "P_value": results_single.pvalues,
                                            "Observations": [results_single.nobs] * len(results_single.params)
                                        })

                                        specification = f"{filter}_{definition}_{measure}_{comparison}_{entropy}_{demo}_{time}_{ER}"
                                        filename_single = f"{outcome_label}_{predictor_label}_single_result_{specification}.csv"
                                        if not df_single.empty:
                                            df_single.to_csv(os.path.join(results_folder, filename_single), index=False)


In [ ]:
# Setup
graph_folder = r"Z:\Projects\EMA_Project\Scripts\Output\Multiverse_Combined_Results"
os.makedirs(graph_folder, exist_ok=True)

csv_files = [f for f in os.listdir(results_folder) if f.endswith(".csv")]
dataframes = {}
for file in csv_files:
    try:
        file_path = os.path.join(results_folder, file)
        dataframes[file] = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file}: {e}")

In [ ]:
# Function
def get_graph_data(outcome_var, predictor_var, dataframes, mode):
    parameter_data = []
    lower_data = []
    upper_data = []
    predictor_labels = []

    suffix = "_single_" if mode == "single" else "_result_"

    for file, data in dataframes.items():
        if f"{outcome_var}_{predictor_var}{suffix}" in file:
            if all(col in data.columns for col in ["Variable", "Parameter", "Std_Err", "Observations"]):
                predictor_row = data[data["Variable"] == predictor_var]
                if not predictor_row.empty:
                    try:
                        parameter_value = predictor_row.iloc[0]["Parameter"]
                        std_error = predictor_row.iloc[0]["Std_Err"]
                        observations = predictor_row.iloc[0]["Observations"]
                        alpha = 0.05 / (np.sqrt(observations / 100))
                        z_score = norm.ppf(1 - (alpha / 2))
                        lower_bound = parameter_value - (z_score * std_error)
                        upper_bound = parameter_value + (z_score * std_error)

                        parameter_data.append(parameter_value)
                        lower_data.append(lower_bound)
                        upper_data.append(upper_bound)
                        predictor_labels.append(file)
                    except Exception as e:
                        print(f"Error processing row in {file}: {e}")

    if parameter_data:
        sorted_indices = np.argsort(parameter_data)
        return (
            np.array(parameter_data)[sorted_indices],
            np.array(lower_data)[sorted_indices],
            np.array(upper_data)[sorted_indices],
            np.array(predictor_labels)[sorted_indices],
        )
    else:
        return [], [], [], []

In [ ]:
# Loop over all combinations
for o_var in ["PA", "NA", "ST"]:
    for p_var in ["RP", "SP", "DS", "SA", "SS"]:
        for mode in ["multi", "single"]:
            parameter_data, lower_data, upper_data, predictor_labels = get_graph_data(o_var, p_var, dataframes, mode=mode)
            if parameter_data.size > 0:
                suffix = "single" if mode == "single" else "multi"
                df = pd.DataFrame({
                    "Parameter": parameter_data,
                    "Lower": lower_data,
                    "Upper": upper_data,
                    "Specification": predictor_labels
                })
                csv_path = os.path.join(graph_folder, f"{o_var}_{p_var}_{suffix}.csv")
                df.to_csv(csv_path, index=False)

In [19]:
# Setup
graph_folder = r"Z:\Projects\EMA_Project\Scripts\Output\Multiverse_Combined_Results_No_Corr"
os.makedirs(graph_folder, exist_ok=True)

csv_files = [f for f in os.listdir(results_folder) if f.endswith(".csv")]
dataframes = {}
for file in csv_files:
    try:
        file_path = os.path.join(results_folder, file)
        dataframes[file] = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file}: {e}")

In [ ]:
# Alternative version of get graph data without alpha correction
# Function
def get_graph_data_no_corr(outcome_var, predictor_var, dataframes, mode):
    parameter_data = []
    lower_data = []
    upper_data = []
    predictor_labels = []

    suffix = "_single_" if mode == "single" else "_result_"

    for file, data in dataframes.items():
        if f"{outcome_var}_{predictor_var}{suffix}" in file:
            if all(col in data.columns for col in ["Variable", "Parameter", "Std_Err", "Observations"]):
                predictor_row = data[data["Variable"] == predictor_var]
                if not predictor_row.empty:
                    try:
                        parameter_value = predictor_row.iloc[0]["Parameter"]
                        std_error = predictor_row.iloc[0]["Std_Err"]
                        observations = predictor_row.iloc[0]["Observations"]
                        alpha = 0.05 # No correction
                        z_score = norm.ppf(1 - (alpha / 2))
                        lower_bound = parameter_value - (z_score * std_error)
                        upper_bound = parameter_value + (z_score * std_error)

                        parameter_data.append(parameter_value)
                        lower_data.append(lower_bound)
                        upper_data.append(upper_bound)
                        predictor_labels.append(file)
                    except Exception as e:
                        print(f"Error processing row in {file}: {e}")

    if parameter_data:
        sorted_indices = np.argsort(parameter_data)
        return (
            np.array(parameter_data)[sorted_indices],
            np.array(lower_data)[sorted_indices],
            np.array(upper_data)[sorted_indices],
            np.array(predictor_labels)[sorted_indices],
        )
    else:
        return [], [], [], []

In [ ]:
# Loop over all combinations
for o_var in ["PA", "NA", "ST"]:
    for p_var in ["RP", "SP", "DS", "SA", "SS"]:
        for mode in ["multi", "single"]:
            parameter_data, lower_data, upper_data, predictor_labels = get_graph_data_no_corr(o_var, p_var, dataframes, mode=mode)
            if parameter_data.size > 0:
                suffix = "single" if mode == "single" else "multi"
                df = pd.DataFrame({
                    "Parameter": parameter_data,
                    "Lower": lower_data,
                    "Upper": upper_data,
                    "Specification": predictor_labels
                })
                csv_path = os.path.join(graph_folder, f"{o_var}_{p_var}_{suffix}.csv")
                df.to_csv(csv_path, index=False)